# Automated promoter library redesign

這本 notebook 從 high-throughput PKL database 選取元件，不進行 de novo sequence generation。

每個 element 的 pooling 邊界由 `CONFIG.mutable_energy_fraction_ranges` 逐元素指定，形式是
`(lower_fraction, upper_fraction, n_bins)` —— 在該元素自己的觀測 min–max energy 軸上，
要納入哪一段、切成幾個等寬 bin。**每個 bin 各出一個 mutable 版本**，所以 N 個 bin 給出
`v1..vN`，再加一個鎖定的 consensus 落在 `v{N+1}`，且 `v1..vN` 必須全部弱於它。

Spacer 是例外：`Spacer_v2` 來自 bin 2，`Spacer_v3 = Spacer_v2[:-2] + "TG"`（衍生而非取自 bin 3），
兩者是同一個 coupled design unit，所以 spacer 至少要 3 個 bin。

每個 design state 組合成 `∏(N_e + 1)` variants；六個元素都是 4 個 bin 時就是
`5^6 = 15,625`（Batch 0 會把實際數字印出來）。使用固定且最大程度均勻的 64 種 3-bp gaps，
再由 CorePromoter clean model 掃描最佳 register。

Validation：

- `shift != 0` 就計入 shifted variant。
- m10 shifted rate 與 m35 shifted rate 都必須 `< 10%`。
- 所有 shifted variants 都必須在 `-2..+2 bp`；任何 `abs(shift) > 2` 都不通過。
- 先最佳化 m10；m10 通過後作為 hard constraint，再最佳化 m35。
- 只有 global validation objective 嚴格改善才接受 replacement。


In [ ]:
# === Batch 0: setup and editable design config ===
import importlib
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

import automated_promoter_library_design as r
importlib.reload(r)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Select the frozen whole/CorePromoter checkpoint used for full-sequence scanning.
CORE_MODEL_VARIANT = "baseline"  # "baseline" or "tss_pas"
CORE_MODEL_CHECKPOINTS = {
    "baseline": r.WEIGHTS_DIR / "weights_CorePromoter_clean.pt",
    "tss_pas": r.WEIGHTS_DIR / "weights_CorePromoter_tss_pas.pt",
}
if CORE_MODEL_VARIANT not in CORE_MODEL_CHECKPOINTS:
    raise ValueError(f"Unknown CORE_MODEL_VARIANT: {CORE_MODEL_VARIANT!r}")
CORE_MODEL_CHECKPOINT = CORE_MODEL_CHECKPOINTS[CORE_MODEL_VARIANT]
if not CORE_MODEL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing {CORE_MODEL_VARIANT} checkpoint: {CORE_MODEL_CHECKPOINT}. "
        "Run Model_CorePromoter_TSS_pretrain.ipynb first."
    )

# RUN_MODE = "new": create a new output directory.
# RUN_MODE = "resume": continue an existing directory from its checkpoint/final_elements.
RUN_MODE = "new"  # "new" or "resume"
RESUME_DIR = r.DEFAULT_PARENT_OUT / "automated_redesign_20260716_112038"

if RUN_MODE == "new":
    OUT_DIR = r.DEFAULT_PARENT_OUT / f"automated_redesign_{RUN_STAMP}"
elif RUN_MODE == "resume":
    OUT_DIR = Path(RESUME_DIR)
    if not OUT_DIR.exists():
        raise FileNotFoundError(f"Resume directory does not exist: {OUT_DIR}")
else:
    raise ValueError(f"RUN_MODE must be 'new' or 'resume', not {RUN_MODE!r}")

CACHE_DIR = r.PROJECT_ROOT / "outputs" / "energy_bin_cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Edit these sequences when the locked consensus changes.
CONSENSUS = {
    "UP": "TGGACTGATATATACAAAA",
    "m35": "TTGACA",
    "spacer": "TATGGCGCAAAATGGGG",
    "m10": "TATAAT",
    "DIS": "TTTTATTA",
    "ITS": "CAAAAAAAAG",
}

# Per-element pooling boundary, as (lower_fraction, upper_fraction, n_bins) on that
# element's own observed min-max energy axis. Read the fractions straight off the
# Batch 0.5 top axis.
#
# Every bin becomes one mutable version, so N bins give v1..vN plus the locked
# consensus at v{N+1}, and one design state assembles prod(N_e + 1) variants.
# All six at 4 bins -> 5^6 = 15,625, printed below so an edit shows up immediately.
#
# Tune the lower bound to trade weak-sequence coverage against register stability,
# and the upper bound against the locked consensus: v1..vN must all be weaker than
# it, so a bin above the red v5 line in Batch 0.5 has no eligible sequence and
# DesignSpace raises instead of silently picking from a neighbouring bin.
# The spacer needs at least 3 bins because Spacer_v3 = Spacer_v2[:-2] + "TG" is
# derived rather than drawn from bin 3.
ENERGY_BIN_RANGES = {
    "UP":     (0.0, 0.8, 4),
    "m35":    (0.3, 0.9, 4),
    "spacer": (0.0, 0.8, 4),
    "m10":    (0.3, 0.9, 4),
    "DIS":    (0.0, 0.8, 4),
    "ITS":    (0.0, 0.8, 4),
}

CONFIG = r.DesignConfig(
    consensus=CONSENSUS,
    bg5="CCCTTTCGTCTTCACACAGCAGCAGTCAGGTAGGGAAGAGACC",
    bg3="GTCGACTCTAGA",
    gap_length=3,
    gap_seed=777,
    random_seed=777,
    n_energy_bins=4,  # fallback bin count for any element omitted from the dict below
    mutable_energy_fraction_ranges=ENERGY_BIN_RANGES,
    max_candidates_per_unit=60,
    max_abs_shift=2,
    max_shift_rate=0.10,
    scan_batch_size=15625,
    require_derived_spacer_in_database=False,
)

USE_CACHE = True
MAX_SOURCE_ROWS = None  # None = use all observed sequences in each PKL

SEARCH_SETTINGS = {
    # This is an additional iteration allowance for each new/resume execution.
    "max_iterations": 100,
    "n_driver_units": 3,
    "probe_candidates_per_unit": 2,
    "pair_beam_width": 6,
    "max_pair_evaluations": 8,
    "max_stalled_iterations": 30,
}

config_filename = "run_config.json" if RUN_MODE == "new" else f"resume_config_{RUN_STAMP}.json"
r.save_run_config(
    OUT_DIR, CONFIG, SEARCH_SETTINGS, DEVICE,
    core_model_checkpoint=CORE_MODEL_CHECKPOINT,
    filename=config_filename,
)
print("Run mode:", RUN_MODE)
print("Project root:", r.PROJECT_ROOT)
print("Device:", DEVICE)
print("Output:", OUT_DIR)
print("Core model variant:", CORE_MODEL_VARIANT)
print("Core model checkpoint:", CORE_MODEL_CHECKPOINT)
print()
print("Energy bins and versions per element:")
for element in r.ELEMENTS:
    n_bins, lower, upper, mode = CONFIG.energy_bin_spec(element)
    print(f"  {element:7s} fraction {lower:.2f}-{upper:.2f} into {n_bins} bins"
          f"  ->  mutable {', '.join(CONFIG.mutable_versions(element))}"
          f"  + locked {CONFIG.locked_version(element)}   [{mode}]")
print("Variants per design state:", f"{CONFIG.n_variants():,}",
      "=", " x ".join(str(len(CONFIG.versions_for(e))) for e in r.ELEMENTS))


## Batch 0.5: 六個文庫的能量分布與現行 pooling 邊界

在 Batch 1 真正切 bin 之前，先看清楚每個文庫的能量分布長什麼樣，用來決定 Batch 0 的
`ENERGY_BIN_RANGES` 每個元素該填什麼。

每個元素的邊界是 `(lower_fraction, upper_fraction, n_bins)`，**每個 bin 各出一個 mutable
版本**（N 個 bin → `v1..vN` + locked `v{N+1}`）。等寬切法對這些分布並不友善：實測全部六個
文庫都嚴重集中，能量全距的兩端非常稀疏。以 spacer 為例，0–0.8 這段切 4 份的計數是
`32,020 / 246,556 / 53,186 / 1,751` —— 相差兩個數量級，而 0.8–1.0 那段整段只有 52 條；
ITS 的 bin 3 一個人吃掉 511,415 條。這就是為什麼邊界要逐元素看圖決定，而不是統一切全距。

這個 cell 呼叫的是 Batch 1 一模一樣的 `build_scored_pools()` / `energy_bin_summary()`，
所以直方圖的橫軸與實際切 bin 的軸是**同一把尺**，不是另外算一套；`build_scored_pools`
有檔案 signature 快取，Batch 1 再跑一次是快取命中。

每個 panel（順序 UP → -35 → spacer 17 → -10 → DIS → ITS）：

- **橫軸** = 該 element model 算出的 energy（higher = stronger）。**每個 panel 各自縮放**，
  不同 element 的分數不可互相比較。上緣副軸是 `energy_fraction` 0–1，也就是要填進
  `ENERGY_BIN_RANGES` 的那個尺度 —— 可以直接從圖上讀出邊界該設多少。
- **縱軸** = 序列數。計數的是**去重後、長度與 ACGT 都合規**的序列，也就是實際進入
  pooling 的那個 pool 的筆數，不是文庫的 read count。預設 log10 軸，否則稀疏尾巴看不見；
  把 `LOG_COUNTS` 改成 `False` 可看原始線性計數。
- **灰色虛線** = 現行 bin 邊界，上緣標出每個 bin 實際裝到幾條序列。
- **淺藍色區塊** = 目前納入切 bin 的能量範圍；灰底區塊是被 `lower/upper_fraction` 排除掉的
  部分（預設下 UP/spacer/DIS/ITS 是 0.8–1.0，m35/m10 是 0–0.3 與 0.9–1.0）。
- **紅線** = locked consensus 的能量。因為 mutable 版本的 hard constraint 是「必須弱於它」，
  這條線決定 `upper_fraction` 往上能拉到哪：**任何超過紅線的 bin 都會抓不到候選而讓 Batch 2
  直接報錯**。表格的 `n_eligible_below_v5` 就是該 bin 扣掉這個限制後真正剩下的條數。

> 幾個一眼可見的陷阱：UP 的紅線在 fraction 0.726，所以 `upper_fraction` 拉到 0.8 以上就開始
> 出現無效 bin（原本 0–1 切 5 份的 bin 5 正是 `n_eligible_below_v5 = 0`）；spacer 的 consensus
> 甚至落在 pool 之外（fraction 1.085，比 SL17 最強的序列還強），m35/m10/DIS 的 consensus 剛好
> 就是 pool 最大值（fraction 1.000）。

-35 / -10 的能量取自 `Model_PL.ipynb` 從 `PL.pkl` 做出來的 BPM 模型
（`BPM/Params_Con17.pkl`，取負號轉成 higher-is-stronger），這正是 pooling 實際用的軸。
`weights_minus35.pt` / `weights_minus10.pt` 雖然在 `weights/` 裡，但 pipeline 沒有任何
code 載入它們（`ElementModelBundle._load_all` 明確跳過），而且與 BPM 排序嚴重不一致
（Spearman ρ = 0.37 / −0.22），所以這裡不使用。


In [ ]:
# === Batch 0.5: per-library energy distributions and current pooling boundaries ===
# English-only plot labels for VSCode/Jupyter rendering stability.
import numpy as np

# log10 y axis. The spacer pool puts 246,556 sequences in bin 2 and only 52 in
# bin 5, so a linear axis hides exactly the sparse tails the boundary decision
# depends on. Set False to read raw linear counts.
LOG_COUNTS = True
SHOW_FRACTION_AXIS = True   # top axis = energy_fraction, the scale that
                            # CONFIG.mutable_energy_fraction_ranges is written in
N_HIST_BINS = 80

PANEL_LABELS = {"UP": "UP", "m35": "-35", "spacer": "spacer 17",
                "m10": "-10", "DIS": "DIS", "ITS": "ITS"}
# -35/-10 come from the BPM model that Model_PL.ipynb derives from PL.pkl
# (BPM/Params_Con17.pkl), negated to a higher-is-stronger axis. The orphaned
# weights_minus35.pt / weights_minus10.pt are deliberately not used here:
# nothing in the pipeline loads them and they disagree with BPM
# (Spearman rho = 0.37 / -0.22), so they cannot explain the bins below.
PANEL_NOTES = {
    "UP": "UL.pkl, 19 bp, NN",
    "m35": "PL.pkl:minus35, 6 bp, BPM -dG",
    "spacer": "SL17.pkl, 17 bp, NN",
    "m10": "PL.pkl:minus10, 6 bp, BPM -dG",
    "DIS": "DL.pkl, 8 bp, NN",
    "ITS": "ITS.pkl, 10 bp, NN",
}

# Same calls Batch 1 makes, so this histogram and the bins it diagnoses share one
# score axis. build_scored_pools is signature-cached, so Batch 1 re-runs for free.
element_models = r.ElementModelBundle(DEVICE)
scored_pools = r.build_scored_pools(
    models=element_models,
    config=CONFIG,
    cache_dir=CACHE_DIR,
    use_cache=USE_CACHE,
    max_source_rows=MAX_SOURCE_ROWS,
)
bin_summary = r.energy_bin_summary(scored_pools, CONFIG, element_models)


def pool_bin_edges(pool):
    """Rebuild the exact bin edges from the constant columns of the scored pool.

    energy_bin_summary leaves energy_lower/energy_upper as NaN for empty bins,
    so the edges are recomputed from energy_min/max and the stored fractions.
    """
    e_min = float(pool["energy_min"].iloc[0])
    e_max = float(pool["energy_max"].iloc[0])
    frac_lo = float(pool["binning_fraction_lower"].iloc[0])
    frac_hi = float(pool["binning_fraction_upper"].iloc[0])
    n_bins = int(pool["n_assigned_bins"].iloc[0])
    span = e_max - e_min
    edges = np.linspace(e_min + frac_lo * span, e_min + frac_hi * span, n_bins + 1)
    return edges, e_min, e_max, span, frac_lo, frac_hi, n_bins


stats_rows = []
fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.6), dpi=140)

for ax, element in zip(axes.ravel(), r.ELEMENTS):
    pool = scored_pools[element]
    energy = pool["energy"].to_numpy(dtype=float)
    edges, e_min, e_max, span, frac_lo, frac_hi, n_bins = pool_bin_edges(pool)
    v5 = float(bin_summary.loc[bin_summary["element"] == element, "v5_energy"].iloc[0])

    # lower_fraction > 0 leaves out-of-range sequences with <NA> energy_bin.
    bin_counts = (
        pool["energy_bin"].dropna().astype(int).value_counts()
        .reindex(range(1, n_bins + 1), fill_value=0).astype(int)
    )

    if frac_lo > 0.0:
        ax.axvspan(e_min, edges[0], color="0.82", alpha=0.6, zorder=0,
                   label="Outside binned range")
    if frac_hi < 1.0:
        ax.axvspan(edges[-1], e_max, color="0.82", alpha=0.6, zorder=0,
                   label=None if frac_lo > 0.0 else "Outside binned range")
    ax.axvspan(edges[0], edges[-1], color="#4C72B0", alpha=0.07, zorder=0,
               label="Binned range")

    ax.hist(energy, bins=N_HIST_BINS, color="#4C72B0", zorder=2)
    for i, edge in enumerate(edges):
        ax.axvline(edge, color="0.30", ls="--", lw=0.9, zorder=3,
                   label="Current bin edge" if i == 0 else None)
    ax.axvline(v5, color="#C44E52", lw=1.8, zorder=4, label="v5 consensus")

    # Headroom above the tallest histogram bar for the per-bin count labels.
    top = max(int(np.histogram(energy, bins=N_HIST_BINS)[0].max()), 1)
    if LOG_COUNTS:
        ax.set_yscale("log")
        ax.set_ylim(0.7, top * 10 ** 0.60)
        label_y = top * 10 ** 0.14
    else:
        ax.set_ylim(0, top * 1.32)
        label_y = top * 1.14
    for bin_id, lo, hi in zip(range(1, n_bins + 1), edges[:-1], edges[1:]):
        # The white backing keeps the v5 line from cutting through the number.
        ax.text((lo + hi) / 2.0, label_y, f"bin {bin_id}\n{bin_counts[bin_id]:,}",
                ha="center", va="bottom", fontsize=7.5, color="0.25", zorder=5,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))

    # v5 can sit outside the observed pool range - the spacer consensus scores
    # 4.66 while the strongest sequence in SL17.pkl only reaches 4.05 - so widen
    # the view to keep that line visible instead of clipping it away.
    view_lo, view_hi = min(e_min, v5), max(e_max, v5)
    pad = 0.02 * (view_hi - view_lo)
    ax.set_xlim(view_lo - pad, view_hi + pad)
    ax.set_title(
        f"{PANEL_LABELS[element]}   ({PANEL_NOTES[element]})\n"
        f"n = {len(pool):,}   |   {n_bins} bins over fraction {frac_lo:.2f}-{frac_hi:.2f}",
        fontsize=9.5, pad=20 if SHOW_FRACTION_AXIS else 6,
    )
    if SHOW_FRACTION_AXIS:
        sec = ax.secondary_xaxis(
            "top",
            functions=(lambda e, o=e_min, s=span: (e - o) / s,
                       lambda f, o=e_min, s=span: o + f * s),
        )
        sec.tick_params(labelsize=7, pad=1)

    fraction = (energy - e_min) / span
    inside = (fraction >= frac_lo - 1e-12) & (fraction <= frac_hi + 1e-12)
    p1, p25, p50, p75, p99 = np.percentile(energy, [1, 25, 50, 75, 99])
    stats_rows.append({
        "element": element,
        "source": PANEL_NOTES[element],
        "n_sequences": int(len(energy)),
        "energy_min": e_min, "p1": p1, "p25": p25, "median": p50,
        "p75": p75, "p99": p99, "energy_max": e_max,
        "binning_mode": str(pool["binning_mode"].iloc[0]),
        "n_bins": n_bins,
        "fraction_lower": frac_lo, "fraction_upper": frac_hi,
        "range_energy_lower": float(edges[0]), "range_energy_upper": float(edges[-1]),
        "n_inside_range": int(inside.sum()),
        "frac_inside_range": float(inside.mean()),
        "v5_energy": v5,
        "v5_energy_fraction": float((v5 - e_min) / span),
        "n_below_v5": int((energy < v5).sum()),
    })

handles, labels = [], []
for ax in axes.ravel():
    for handle, label in zip(*ax.get_legend_handles_labels()):
        if label not in labels:
            handles.append(handle)
            labels.append(label)

fig.suptitle("Per-library energy distributions and current pooling boundaries",
             fontsize=14, y=0.995)
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.945),
           ncol=len(labels), frameon=False, fontsize=8.5)
fig.supxlabel("Element model energy, higher = stronger "
              "(per-panel scale; not comparable across elements). "
              "Top axis = energy_fraction 0-1.", fontsize=10)
fig.supylabel(f"Sequence count{' (log10)' if LOG_COUNTS else ''}", fontsize=11)
fig.tight_layout(rect=(0.012, 0.015, 1, 0.905))
for ext in ("png", "svg"):
    fig.savefig(OUT_DIR / f"element_energy_distributions.{ext}", bbox_inches="tight")
plt.show()

element_energy_stats = pd.DataFrame(stats_rows)
pool_sizes = {element: len(scored_pools[element]) for element in r.ELEMENTS}
element_energy_bin_counts = bin_summary.copy()
element_energy_bin_counts["pct_of_pool"] = (
    100.0 * element_energy_bin_counts["n_sequences"]
    / element_energy_bin_counts["element"].map(pool_sizes)
)

element_energy_stats.to_csv(OUT_DIR / "element_energy_distribution_stats.csv", index=False)
element_energy_bin_counts.to_csv(OUT_DIR / "element_energy_bin_counts.csv", index=False)
display(element_energy_stats.round(4))
display(element_energy_bin_counts.round(4))
print("Figure and tables written to:", OUT_DIR)


## Batch 1: load trained models and build the per-element energy bins

UP/Spacer/DIS/ITS 使用各自已訓練的 element weights。`Model_PL.ipynb` 的 -35/-10 data flow 使用 BPM，因此這兩個元素使用 `-BPM dG` 作為 higher-is-stronger score。不同 element 的分數不可互相比較。

切幾個 bin、切在哪一段，完全由 Batch 0 的 `ENERGY_BIN_RANGES` 決定（見 Batch 0.5 的分布圖）。
`energy_bin_summary` 逐 bin 列出 `n_sequences` 與 `n_eligible_below_v5` —— 後者才是真正可用的
候選數，因為 mutable 版本必須弱於 locked consensus；某個 bin 的 `n_eligible_below_v5` 為 0 時，
Batch 2 會直接報錯而不是跨 bin 補候選。

Scored pools 會依 PKL 與 weight/BPM parameter 的檔案 signature 快取；來源或模型更新後會自動重建。改動 bin 邊界不需要重算能量，快取仍然有效。


In [ ]:
element_models = r.ElementModelBundle(DEVICE)
core_model = r.load_core_model(DEVICE, checkpoint_path=CORE_MODEL_CHECKPOINT)

scored_pools = r.build_scored_pools(
    models=element_models,
    config=CONFIG,
    cache_dir=CACHE_DIR,
    use_cache=USE_CACHE,
    max_source_rows=MAX_SOURCE_ROWS,
)

bin_summary = r.energy_bin_summary(scored_pools, CONFIG, element_models)
bin_summary.to_csv(OUT_DIR / "energy_bin_summary.csv", index=False)
display(bin_summary)


## Batch 2: construct database-backed design units

這個步驟執行 hard constraints：

- 一般 element 的 `v1..vN` 必須各自來自對應的 bin 1..N（N = 該元素的 `n_bins`）。
- 所有 mutable 版本的 score 必須低於 locked consensus（`v{N+1}`）。
- 同一 element 的各條 sequence 不可重複。
- Spacer_v2/v3 必須滿足 coupled rule（v3 = v2[:-2] + "TG"，不取自 bin 3）。
- 任一必要 bin 沒有 eligible sequence 時直接報錯，不跨 bin 補候選 —— 這通常表示該 bin 的上界
  設得比 locked consensus 還強，回 Batch 0 把 `upper_fraction` 調低。


In [ ]:
design_space = r.DesignSpace(
    config=CONFIG,
    models=element_models,
    scored_pools=scored_pools,
)

if RUN_MODE == "resume":
    saved_elements_path = OUT_DIR / "current_elements_checkpoint.csv"
    if not saved_elements_path.exists():
        saved_elements_path = OUT_DIR / "final_elements.csv"
    if not saved_elements_path.exists():
        raise FileNotFoundError(f"No resume elements found in {OUT_DIR}")
    initial_elements = pd.read_csv(saved_elements_path)
    initial_state = design_space.state_from_selected_elements(initial_elements)
    print("Recovered state from:", saved_elements_path)
else:
    initial_state = design_space.initial_state()
    initial_elements = design_space.selected_elements(initial_state)

unit_summary = design_space.candidate_pool_summary()

if RUN_MODE == "new":
    initial_elements.to_csv(OUT_DIR / "initial_elements.csv", index=False)
unit_summary.to_csv(OUT_DIR / "design_unit_candidate_counts.csv", index=False)
display(initial_elements)
display(unit_summary)
print("Candidate pool summary before search:")
print(unit_summary.to_string(index=False))


## Batch 3: fixed balanced 3-bp gap assignment and library assembly

Variant 數是 `∏(N_e + 1)`，由 Batch 0 的 bin 數決定；六個元素都是 4 個 bin 時為 15,625。

64 種 3-bp gap 要盡量平均分配到這些 variants 上。`15,625 = 64 × 244 + 9`，無法完全等量，
所以最大程度均勻的固定分配是 55 種 gap 各 244 次、9 種各 245 次（`build_balanced_gap_assignment`
會斷言任兩種 gap 的次數差不超過 1，因此改變 bin 數之後這個分配仍然成立，只是餘數不同）。
這份 assignment 在所有 replacement 前後保持不變。


In [ ]:
saved_gap_path = OUT_DIR / "gap_assignment.csv"
if RUN_MODE == "resume" and saved_gap_path.exists():
    gap_assignment = pd.read_csv(saved_gap_path)
    print("Loaded fixed gap assignment:", saved_gap_path)
else:
    gap_assignment = r.build_balanced_gap_assignment(CONFIG)

initial_variants = r.assemble_library(initial_elements, gap_assignment, CONFIG)

gap_counts = (
    gap_assignment["gap_3bp"]
    .value_counts()
    .rename_axis("gap_3bp")
    .reset_index(name="n_variants")
    .sort_values("gap_3bp")
)
gap_assignment.to_csv(OUT_DIR / "gap_assignment.csv", index=False)
if RUN_MODE == "new":
    initial_variants.to_csv(OUT_DIR / f"initial_assembled_{CONFIG.n_variants()}.csv", index=False)

print("Assembled variants:", f"{len(initial_variants):,}")
print("Gap count distribution:", gap_counts["n_variants"].value_counts().sort_index().to_dict())
display(gap_counts)


## Batch 4: initial CorePromoter register scan

正式欄位：

```text
m10_shift = observed_m10_start - design_m10_start
m35_shift = observed_m35_start - design_m35_start
spacer_length_shift = observed_spacer_len - design_spacer_len
```

Primary classification 使用 `m10_shift`；m10 通過後，secondary phase 使用 `m35_shift`。掃描同時輸出八個 architecture windows、observed element-role sequences、落在哪些 design regions，以及同一 element model 內的 delta energy。


In [ ]:
scanner = r.CorePromoterScanner(
    model=core_model,
    element_models=element_models,
    config=CONFIG,
    device=DEVICE,
)

initial_scan = scanner.scan(initial_variants, annotate_element_energies=True)
initial_summary = r.summarize_validation(initial_scan, CONFIG)
initial_scan.to_csv(OUT_DIR / f"initial_scan_{CONFIG.n_variants()}.csv", index=False)

validation_table = pd.DataFrame([
    {
        "anchor": anchor,
        "shifted_count": initial_summary[f"{anchor}_shifted_count"],
        "shifted_rate": initial_summary[f"{anchor}_shifted_rate"],
        "out_of_range_count": initial_summary[f"{anchor}_out_of_range_count"],
        "max_abs_shift": initial_summary[f"{anchor}_max_abs_shift"],
        "pass": initial_summary[f"{anchor}_validation_pass"],
    }
    for anchor in ("m10", "m35")
])
display(validation_table)
print("Global validation pass:", initial_summary["global_validation_pass"])


In [ ]:
# English-only plot labels for VSCode/Jupyter rendering stability.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=140)
for ax, anchor in zip(axes, ("m10", "m35")):
    counts = initial_scan[f"{anchor}_shift"].value_counts().sort_index()
    ax.bar(counts.index.astype(int), counts.values, color="#4C72B0")
    # ax.axvspan(-2, 2, color="#55A868", alpha=0.15, label="Allowed magnitude")
    ax.set_title(f"{anchor} shift distribution")
    ax.set_xlabel("Shift (bp)")
    ax.set_ylabel("Variant count")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Batch 5: dominant shift and conditional-risk diagnosis

若目前 phase 仍有 `abs(shift) > 2`，優先診斷 out-of-range variants；否則診斷所有 `shift != 0` variants。先找數量最多的 shift coordinate，再以 `P(dominant shift | element version)` 排名 driver。

Observed role 落在某個 design region 只用來提供 redesign evidence；不同 element model 的 raw delta energy不互相比較。最終 replacement 仍由完整 15,625 before/after validation 決定。


In [ ]:
initial_diagnosis = r.diagnose_shift_drivers(
    scan=initial_scan,
    selected_elements=initial_elements,
    design_space=design_space,
    config=CONFIG,
)

print("Optimization phase:", initial_diagnosis["phase"])
print("Dominant shift coordinate:", initial_diagnosis["dominant_shift"])
display(initial_diagnosis["risk"].head(20))
display(initial_diagnosis["overlap"].head(30))


## Batch 6: monotonic automated redesign with live progress and resume

每輪會 print iteration、phase、目前 m10/m35 shifted rate、out-of-range count、dominant shift、driver units、每個 proposal 結果與接受/拒絕原因。

執行期間會在 `OUT_DIR` 持續覆寫：

- `search_progress.csv`：每輪狀態的持續更新 DataFrame。
- `proposal_history_checkpoint.csv`：所有已測 single/double proposals。
- `current_elements_checkpoint.csv`：目前接受的 30 條 sequences。
- `current_validation.json`：目前 validation。
- `search_checkpoint.json`：state、最後完成輪數、stalled counter 與已測 proposals。

`progress_df` 也會在記憶體中每輪原地更新；手動 interrupt 後可直接 `display(progress_df.tail())`。

若中途停止 kernel，將 setup cell 的 `RUN_MODE` 改為 `"resume"`，並把 `RESUME_DIR` 指向原本的 output directory；Batch 6 會從下一個 iteration 繼續。每次 resume 的 `max_iterations` 是額外輪數，不是總累積上限。


In [ ]:
redesigner = r.AutomatedRedesigner(
    design_space=design_space,
    scanner=scanner,
    gap_assignment=gap_assignment,
    config=CONFIG,
    out_dir=OUT_DIR,
    **SEARCH_SETTINGS,
)

# Mutated in place after every completed iteration. It remains available if
# this cell is manually interrupted.
progress_df = pd.DataFrame()

result = redesigner.run(
    initial_state=initial_state,
    initial_evaluation=(initial_elements, initial_scan, initial_summary),
    resume=(RUN_MODE == "resume"),
    reset_stalled_on_resume=True,
    progress_df=progress_df,
    verbose=False,
)

proposal_df = result.proposals

print("Success:", result.success)
print("Stop reason:", result.stop_reason)
print("Saved to:", result.out_dir)
display(pd.DataFrame([result.final_summary]))
display(progress_df)


## Batch 7: final design and audit tables

`final_elements.csv` 是最後保留的 30 條 element sequences；`proposal_history.csv` 同時保留 accepted 與 rejected moves，方便追蹤為何某次替換沒有被採用。


In [ ]:
display(result.final_elements)
display(result.final_risk.head(20))
display(result.final_overlap.head(30))

final_shift_table = pd.DataFrame([
    {
        "anchor": anchor,
        "shifted_count": result.final_summary[f"{anchor}_shifted_count"],
        "shifted_rate": result.final_summary[f"{anchor}_shifted_rate"],
        "out_of_range_count": result.final_summary[f"{anchor}_out_of_range_count"],
        "max_abs_shift": result.final_summary[f"{anchor}_max_abs_shift"],
        "pass": result.final_summary[f"{anchor}_validation_pass"],
    }
    for anchor in ("m10", "m35")
])
display(final_shift_table)

# English-only plot labels for VSCode/Jupyter rendering stability.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), dpi=140)
for ax, anchor in zip(axes, ("m10", "m35")):
    counts = result.final_scan[f"{anchor}_shift"].value_counts().sort_index()
    ax.bar(counts.index.astype(int), counts.values, color="#4C72B0")
    # ax.axvspan(-2, 2, color="#55A868", alpha=0.15, label="Allowed magnitude")
    ax.set_title(f"Final {anchor} shift distribution")
    ax.set_xlabel("Shift (bp)")
    ax.set_ylabel("Variant count")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

accepted = result.proposals[result.proposals.get("accepted", False) == True] if len(result.proposals) else result.proposals
display(accepted)
